In [2]:
import torch
import pickle
import json
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from deep_translator import GoogleTranslator
from tqdm import tqdm
from collections import defaultdict
import ast

with open("knowledge_graph.pkl", "rb") as f:
    G = pickle.load(f)

with open("graph_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open("Dataset/release_evidences.json", "r", encoding="utf-8") as f:
    evidences = json.load(f)

with open("Dataset/release_conditions.json", "r", encoding="utf-8") as f:
    conditions = json.load(f)

import pandas as pd
df = pd.read_csv("Dataset/release_train_patients")

diagnoses = metadata["diagnoses"]
symptoms  = metadata["symptoms"]

print(f"Граф: {G.number_of_nodes()} вузлів, {G.number_of_edges()} ребер")
print(f"Діагнозів: {len(diagnoses)}")
print(f"Симптомів: {len(symptoms)}")
print(f"Пацієнтів: {len(df):,}")

D:\Maga\Диплом_Мага\ClinicalAdaptiveScenarioGenerator\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Граф: 272 вузлів, 888 ребер
✅ Діагнозів: 49
✅ Симптомів: 223
✅ Пацієнтів: 1,025,602


In [3]:
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

print(f"BioBERT завантажено. Пристрій: {device}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 33086.24it/s]
BertModel LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BioBERT завантажено. Пристрій: cpu


In [4]:
def encode_text(text: str) -> np.ndarray:
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=64,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    cls_vector = outputs.last_hidden_state[:, 0, :]
    return cls_vector.cpu().numpy().squeeze()


def normalize(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / norms


def translate_to_english(text: str) -> str:
    try:
        return GoogleTranslator(source='auto', target='en').translate(text)
    except:
        return text


print("Функції визначено")

✅ Функції визначено


In [5]:
symptom_names = {}
for code, info in evidences.items():
    question = info.get("question_en", "")
    if question:
        label = (question
                 .replace("Do you have ", "")
                 .replace("Have you had ", "")
                 .replace("Are you ", "")
                 .replace("Did you ", "")
                 .replace("Is there ", "")
                 .replace("Do you ", "")
                 .replace("Have you ", "")
                 .rstrip("?")
                 .strip())
        if len(label) > 50:
            label = label[:47] + "..."
        symptom_names[code] = label
    else:
        symptom_names[code] = code

print(f"symptom_names: {len(symptom_names)} записів")

✅ symptom_names: 223 записів


In [6]:
co_occurrence  = defaultdict(lambda: defaultdict(int))
diagnosis_count = defaultdict(int)

print("Розрахунок частот (виправлена версія)...")

for _, row in df.iterrows():
    diagnosis = row["PATHOLOGY"]
    diagnosis_count[diagnosis] += 1

    try:
        evid_list = ast.literal_eval(row["EVIDENCES"])
    except:
        continue

    seen_base_codes = set()
    for ev_code in evid_list:
        base_code = ev_code.split("_@_")[0]
        seen_base_codes.add(base_code)

    for base_code in seen_base_codes:
        ev_name = symptom_names.get(base_code, base_code)
        co_occurrence[diagnosis][ev_name] += 1

for diagnosis, symp_freq in co_occurrence.items():
    total = diagnosis_count[diagnosis]
    for symptom, count in symp_freq.items():
        alpha_ij = count / total
        if G.has_edge(diagnosis, symptom):
            G[diagnosis][symptom]["weight"] = alpha_ij
        elif G.has_node(diagnosis):
            G.add_edge(diagnosis, symptom, weight=alpha_ij, edge_type="frequency")

# Перевірка
example = list(conditions.keys())[0]
top5 = sorted(G[example].items(), key=lambda x: x[1]["weight"], reverse=True)[:5]
print(f"\nТоп-5 для '{example}':")
for symp, data in top5:
    print(f"  α={data['weight']:.3f}  →  {symp}")

max_w = max(data["weight"] for _, _, data in G.edges(data=True))
print(f"\nМакс. вага в графі: {max_w:.3f}  (має бути ≤ 1.0)")

Розрахунок частот (виправлена версія)...

Топ-5 для 'Spontaneous pneumothorax':
  α=1.000  →  feel pain somewhere
  α=1.000  →  pain somewhere, related to your reason for cons...
  α=1.000  →  Does the pain radiate to another location
  α=1.000  →  Characterize your pain:
  α=1.000  →  How fast did the pain appear

Макс. вага в графі: 1.000  (має бути ≤ 1.0)


In [7]:
all_nodes    = diagnoses + symptoms
node_vectors = {}

print(f"Векторизація {len(all_nodes)} вузлів...")

for node_name in tqdm(all_nodes):
    node_vectors[node_name] = encode_text(node_name)

print(f"Векторизовано: {len(node_vectors)} вузлів")

Векторизація 272 вузлів...


100%|██████████| 272/272 [00:05<00:00, 53.76it/s]

✅ Векторизовано: 272 вузлів


In [8]:
diagnosis_matrix = normalize(np.stack([node_vectors[d] for d in diagnoses]))
symptom_matrix   = normalize(np.stack([node_vectors[s] for s in symptoms]))

np.save("diagnosis_vectors.npy", diagnosis_matrix)
np.save("symptom_vectors.npy",   symptom_matrix)

with open("node_vectors.pkl", "wb") as f:
    pickle.dump(node_vectors, f)

print(f"diagnosis_vectors.npy  →  shape: {diagnosis_matrix.shape}")
print(f"symptom_vectors.npy    →  shape: {symptom_matrix.shape}")
print(f"node_vectors.pkl збережено")

✅ diagnosis_vectors.npy  →  shape: (49, 768)
✅ symptom_vectors.npy    →  shape: (223, 768)
✅ node_vectors.pkl збережено


In [9]:
print("Будуємо індекс назв вузлів...")

symptom_name_vectors = {}
for symp in symptoms:
    symptom_name_vectors[symp] = encode_text(symp)

symptom_name_matrix = normalize(
    np.stack([symptom_name_vectors[s] for s in symptoms])
)

np.save("symptom_name_vectors.npy", symptom_name_matrix)
print(f"   symptom_name_vectors.npy  →  shape: {symptom_name_matrix.shape}")

Будуємо індекс назв вузлів...
✅ symptom_name_vectors.npy  →  shape: (223, 768)


In [10]:
adj_matrix = np.zeros((len(diagnoses), len(symptoms)))
for i, diag in enumerate(diagnoses):
    for j, symp in enumerate(symptoms):
        if G.has_edge(diag, symp):
            adj_matrix[i][j] = G[diag][symp]["weight"]

np.save("adjacency_matrix.npy", adj_matrix)

with open("knowledge_graph.pkl", "wb") as f:
    pickle.dump(G, f)

print(f"   adjacency_matrix.npy  →  shape: {adj_matrix.shape}")
print(f"   max α = {adj_matrix.max():.3f}  (має бути ≤ 1.0)")
print(f"   knowledge_graph.pkl оновлено")

✅ adjacency_matrix.npy  →  shape: (49, 223)
   max α = 1.000  (має бути ≤ 1.0)
✅ knowledge_graph.pkl оновлено


In [11]:
def find_similar_nodes(query: str, top_k: int = 5) -> list:
    eng = translate_to_english(query)
    vec = normalize(encode_text(eng).reshape(1, -1))
    sims = cosine_similarity(vec, symptom_name_matrix)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(symptoms[i], float(sims[i])) for i in top_idx]


test_queries = [
    "задишка",
    "біль у грудях",
    "кашель",
    "fever and chills",
    "chest pain and shortness of breath",
]

print("Фінальний тест пошуку:")
print("=" * 60)
for q in test_queries:
    results = find_similar_nodes(q, top_k=3)
    eng = translate_to_english(q)
    print(f"\n'{q}' → '{eng}'")
    for node, score in results:
        bar = "█" * int(score * 20)
        print(f"  {score:.3f} {bar}  →  {node}")

Фінальний тест пошуку:

'задишка' → 'dyspnea'
  0.939 ██████████████████  →  chronic pancreatitis
  0.924 ██████████████████  →  diarrhea or an increase in stool frequency
  0.923 ██████████████████  →  a sore throat

'біль у грудях' → 'chest pain'
  0.940 ██████████████████  →  a sore throat
  0.938 ██████████████████  →  heart failure
  0.936 ██████████████████  →  chest pain even at rest

'кашель' → 'cough'
  0.926 ██████████████████  →  vomit after coughing
  0.925 ██████████████████  →  a cough
  0.914 ██████████████████  →  Are the symptoms or pain increased with coughin...

'fever and chills' → 'fever and chills'
  0.944 ██████████████████  →  chills or shivers
  0.912 ██████████████████  →  diarrhea or an increase in stool frequency
  0.909 ██████████████████  →  chronic pancreatitis

'chest pain and shortness of breath' → 'chest pain and shortness of breath'
  0.933 ██████████████████  →  chest pain even at rest
  0.928 ██████████████████  →  nasal congestion or a clear runny 